In [1]:
import sys
import os
import subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    !pip install -U ipython
    !git clone https://github.com/AdrianPanasiewicz/QML_for_radar_classification.git

    repo_url = "https://github.com/AdrianPanasiewicz/QML_for_radar_classification.git"
    repo_path = "/content/QML_for_radar_classification"
    colab_run_dir = os.path.join(repo_path, 'colab_run')

    def run(cmd, cwd=None):
        return subprocess.check_output(cmd, cwd=cwd, text=True).strip()

    if not os.path.isdir(os.path.join(repo_path, ".git")):
        run(["git", "clone", repo_url, repo_path])
    else:
        run(["git", "fetch", "origin"], cwd=repo_path)
        local_head = run(["git", "rev-parse", "HEAD"], cwd=repo_path)
        remote_head = run(["git", "rev-parse", "origin/HEAD"], cwd=repo_path)
        if local_head != remote_head:
            run(["git", "reset", "--hard", "origin/HEAD"], cwd=repo_path)

    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)


    os.makedirs(colab_run_dir, exist_ok=True)
    os.chdir(colab_run_dir)

    !pip install -q pennylane
    !pip install "ray[tune]"


%load_ext autoreload
%autoreload 2

%aimport -torch
%aimport -numpy
%aimport -qiskit
%aimport -pennylane
%aimport -ray
%aimport -sklearn

# These import from Data folder are necessary for pickle load to work
from Data.Primitives.environment_classes import Drone, Radar, Context
from Data.Primitives.noise_models import AdditiveWhiteGaussianNoise
from Data.Primitives.presets import *
from Data.Generators.synthetic_dataset_generator import DatasetMetadata, DataRequest

from MachineLearning.Processing.file_loader import SyntheticDataFileLoader
from MachineLearning.Processing.frequency_domain_parser import FrequencyDomainDataParser
from MachineLearning.Processing.time_domain_parser import TimeDomainDataParser
from MachineLearning.Torch_datasets.synthetic_time_dataset import SyntheticTimeDomainRadarDataset
from MachineLearning.Torch_datasets.synthetic_frequency_dataset import SyntheticFrequencyDomainRadarDataset
from MachineLearning.Models.experiment_pure.classical_neural_network import ClassicalNeuralNetwork
from MachineLearning.Models.experiment_pure.quantum_neural_network import QuantumNeuralNetwork
from MachineLearning.Processing.data_visualizer import DataVisualizer
from MachineLearning.Trainers.statistical_trainer import TrainerForModelStatistics
from MachineLearning.Trainers.hyperparameter_trainer import TrainerForHyperparameterSearch

from matplotlib import pyplot as plt
import numpy as np

from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import sympy

import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
from torch.optim import SGD

import ray
from ray import tune
from ray.tune import Checkpoint
from ray.tune.schedulers import ASHAScheduler

MODEL_REGISTRY = {
    "ClassicalNeuralNetwork": ClassicalNeuralNetwork,
}

### Checking preprocessing functionalities

In [ ]:
PROJECT_ROOT = Path().cwd().parent
type = "time_domain"
load_path = PROJECT_ROOT / "Data" / "Datasets" / type / "training_dataset.pkl"

md = DatasetMetadata.create_from_path(load_path)
synt_dataset = SyntheticDataFileLoader(dataset_metadata=md)

obj = synt_dataset.peek_sample(index=6000)

td_data_parser = TimeDomainDataParser()
signal, label, misc_data = td_data_parser.extract_training_data_and_label(obj)
td_data_parser.plot_drone_spectrogram(signal, misc_data, nperseg=32, noverlap=16)

In [ ]:
parsed_signal, label, misc_data = td_data_parser.parse_data_object(obj)
parsed_signal, label, misc_data

In [ ]:
PROJECT_ROOT = Path().cwd().parent
type = "frequency_domain"
load_path = PROJECT_ROOT / "Data" / "Datasets" / type / "training_dataset.pkl"

md = DatasetMetadata.create_from_path(load_path)
synt_dataset = SyntheticDataFileLoader(dataset_metadata=md)

obj = synt_dataset.peek_sample(index=6000)

td_data_parser = FrequencyDomainDataParser()
signal, label, misc_data = td_data_parser.parse_data_object(obj, bin_size=1, return_mag=False)
print(signal.shape)

### Hyperparameter learner

In [2]:
# Safely import the colab module
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = Path().cwd().parent
ray.shutdown()

ctx = ray.init(
    _metrics_export_port=8080,
    runtime_env={
        "working_dir": str(PROJECT_ROOT),
        "excludes": [
            "Data/Datasets"
        ]
    }
)

if IN_COLAB:
    print("Loading Ray Dashboard:")
    output.serve_kernel_port_as_iframe(8265, height=600)

    print("Loading Ray Metrics Export:")
    output.serve_kernel_port_as_iframe(8080, height=400)
else:
    print(f"Running locally. Dashboard available at: {ctx.dashboard_url}")
    print("Metrics available at: http://127.0.0.1:8080")

2026-04-21 09:52:51,602	INFO worker.py:2012 -- Started a local Ray instance.
2026-04-21 09:52:51,607	WARNING working_dir.py:86 -- Directory '.git' is now ignored by default when packaging the working directory. To disable this behavior, set the `RAY_OVERRIDE_RUNTIME_ENV_DEFAULT_EXCLUDES=''` environment variable.
2026-04-21 09:52:51,621	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/content/QML_for_radar_classification/.idea/.gitignore')]
2026-04-21 09:52:51,626	INFO packaging.py:691 -- Creating a file package for local module '/content/QML_for_radar_classification'.
2026-04-21 09:52:51,639	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/content/QML_for_radar_classification/.idea/.gitignore')]
2026-04-21 09:52:51,644	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_596df747d16d769d.zip' (0.58MiB) to Ray cluster...
2026-04-21 09:52:51,648	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray

Loading Ray Dashboard:


/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


<IPython.core.display.Javascript object>

Loading Ray Metrics Export:


<IPython.core.display.Javascript object>

In [ ]:
config_params = 128
divs_array = sympy.divisors(config_params)

pair_map = {div : config_params // div for div in divs_array}

config = {
    "layers": tune.grid_search(list(pair_map.keys())),
    "neurons_per_layer": tune.sample_from(lambda config: pair_map[config["layers"]]),
    "lr": tune.loguniform(1e-5, 1e-3),
    "batch_size": tune.choice([2, 4, 8, 16]),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    'epochs' : 250
}


max_num_epochs = 250
num_trials =  10
scheduler = ASHAScheduler(
    max_t=max_num_epochs,
    grace_period=50,
    reduction_factor=2,
)

cpus_per_trial = 2
gpus_per_trial = 1

PROJECT_ROOT = Path().cwd().parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"
testing_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "testing_dataset.pkl"

trainer = TrainerForHyperparameterSearch(
    training_path=training_path,
    validating_path=validating_path,
    testing_path=testing_path,
    criterion = nn.CrossEntropyLoss()
)

tuner = tune.Tuner(
    tune.with_resources(
        tune.with_parameters(trainer.train_model, model_class=ClassicalNeuralNetwork),
        resources={"cpu": cpus_per_trial, "gpu": gpus_per_trial}
    ),
    tune_config=tune.TuneConfig(
        metric="loss",
        mode="min",
        scheduler=scheduler,
        num_samples=num_trials,
        trial_dirname_creator=lambda trial: f"t_{trial.trial_id}"
    ),
    param_space=config,
)
results = tuner.fit()

+--------------------------------------------------------------------+
| Configuration for experiment     train_model_2026-04-21_09-52-55   |
+--------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator             |
| Scheduler                        AsyncHyperBandScheduler           |
| Number of trials                 80                                |
+--------------------------------------------------------------------+

View detailed results here: /root/ray_results/train_model_2026-04-21_09-52-55
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-04-21_09-52-40_123787_444/artifacts/2026-04-21_09-52-56/train_model_2026-04-21_09-52-55/driver_artifacts`

Trial status: 80 PENDING
Current time: 2026-04-21 09:53:22. Total running time: 25s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+--------------------------------------------------------------

(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000000)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000001)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000002)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000003)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000004)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_mod


Trial status: 1 RUNNING | 79 PENDING
Current time: 2026-04-21 09:53:52. Total running time: 55s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
Current best trial: df4fb_00000 with loss=0.7138102650642395 and params={'layers': 1, 'neurons_per_layer': 128, 'lr': 0.011644857600087242, 'batch_size': 4, 'device': 'cuda', 'epochs': 250}
+-------------------------------------------------------------------------------------------------------------------------------+
| Trial name                status              lr     batch_size     layers     iter     total time (s)      loss     accuracy |
+-------------------------------------------------------------------------------------------------------------------------------+
| train_model_df4fb_00000   RUNNING    0.0116449                4          1       33            18.3506   0.71381          0.5 |
| train_model_df4fb_00001   PENDING    5.41015e-06             16          2                                        

(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000033)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000034)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000035)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000036)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00000/checkpoint_000037)
(train_model pid=1200) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_mod

Trial status: 1 RUNNING | 79 PENDING
Current time: 2026-04-21 09:54:22. Total running time: 1min 25s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
Current best trial: df4fb_00000 with loss=0.6934423446655273 and params={'layers': 1, 'neurons_per_layer': 128, 'lr': 0.011644857600087242, 'batch_size': 4, 'device': 'cuda', 'epochs': 250}
+--------------------------------------------------------------------------------------------------------------------------------+
| Trial name                status              lr     batch_size     layers     iter     total time (s)       loss     accuracy |
+--------------------------------------------------------------------------------------------------------------------------------+
| train_model_df4fb_00000   RUNNING    0.0116449                4          1       98            47.8021   0.693442          0.5 |
| train_model_df4fb_00001   PENDING    5.41015e-06             16          2                                

...

Trial status: 2 TERMINATED | 1 RUNNING | 77 PENDING
Current time: 2026-04-21 09:58:22. Total running time: 5min 25s
Logical resource usage: 2.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
Current best trial: df4fb_00001 with loss=0.6922613978385925 and params={'layers': 2, 'neurons_per_layer': 64, 'lr': 5.410150815947045e-06, 'batch_size': 16, 'device': 'cuda', 'epochs': 250}
+----------------------------------------------------------------------------------------------------------------------------------+
| Trial name                status                lr     batch_size     layers     iter     total time (s)       loss     accuracy |
+----------------------------------------------------------------------------------------------------------------------------------+
| train_model_df4fb_00002   RUNNING      0.00747235               4          4      180           109.604    0.693443     0.5      |
| train_model_df4fb_00000   TERMINATED   0.0116449                4          1      

(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00002/checkpoint_000180)
(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00002/checkpoint_000181)
(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00002/checkpoint_000182)
(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00002/checkpoint_000183)
(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00002/checkpoint_000184)
(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_mod


Trial train_model_df4fb_00002 completed after 200 iterations at 2026-04-21 09:58:34. Total running time: 5min 37s
+------------------------------------------------------------+
| Trial train_model_df4fb_00002 result                       |
+------------------------------------------------------------+
| checkpoint_dir_name                      checkpoint_000199 |
| time_this_iter_s                                   0.58365 |
| time_total_s                                     121.62717 |
| training_iteration                                     200 |
| accuracy                                               0.5 |
| loss                                     0.693248987197876 |
+------------------------------------------------------------+


(train_model pid=2052) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/root/ray_results/train_model_2026-04-21_09-52-55/t_df4fb_00002/checkpoint_000199)



Trial train_model_df4fb_00003 started with configuration:
+--------------------------------------------------+
| Trial train_model_df4fb_00003 config             |
+--------------------------------------------------+
| batch_size                                     2 |
| device                                      cuda |
| epochs                                       250 |
| layers                                         8 |
| lr                                       0.00019 |
| neurons_per_layer                             16 |
+--------------------------------------------------+


### Statistical learner

In [2]:
def l1_normalize_1d(x):
    return normalize(x, p=1, dim=0)

def l2_normalize_1d(x):
    return normalize(x, dim=0)

In [ ]:
PROJECT_ROOT = Path().cwd().parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"
testing_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "testing_dataset.pkl"


config = {
    "layers": 2,
    "neurons_per_layer": 250,
    "lr": 1e-4,
    "batch_size": 16,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    'epochs' : 100
}

trainer = TrainerForModelStatistics(training_path, validating_path, testing_path, criterion = nn.CrossEntropyLoss())
data_array_all_runs = trainer.train_model(ClassicalNeuralNetwork, config, number_of_trials=10, number_of_epochs=500)

Model runs:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
data_array = torch.tensor(
    [data_array_all_runs[i]['accuracy'] for i in range(len(data_array_all_runs))],
    dtype=torch.float32
)

plotter = DataVisualizer()
means, stds = plotter.calculate_statistics(data_array)
plotter.plot_statistics(means, stds)

### Quantum Neural Network

In [ ]:
batch_size = 64
num_qubits = 10

PROJECT_ROOT = Path().cwd().parent
training_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "training_dataset.pkl"
validating_path = PROJECT_ROOT / "Data" / "Datasets" / "time_domain" / "validating_dataset.pkl"

training_data = SyntheticTimeDomainRadarDataset(training_path)
validating_data = SyntheticTimeDomainRadarDataset(validating_path, mean=training_data.mean, std=training_data.std)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
validating_dataloader = DataLoader(validating_data, batch_size=64, shuffle=True)

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
quantum_model = QuantumNeuralNetwork(num_qubits).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(quantum_model.parameters(), lr=1e-3)

trainer = TrainerForHyperparameterSearch(
    training_dataloader=train_dataloader,
    validating_dataloader=validating_dataloader,
    testing_dataloader=None,
    loss_fn=criterion,
)

epochs = 10
accuracy_array=[]
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    trainer.train()
    acc = trainer.test()
    accuracy_array.append(acc)
print("Done!")